# 05G — Global Evaluation (Phase G3)

Evaluation of the global feature explanations (`results/global/`) against the
structured ground truth (`explanations/global_groundtruth/`):

1. **Deterministic rubric** (`utils.rubric`, no LLM) — 0–1 sub-scores on the GT fields
   (direction/monotonicity, importance rank, key structure).
2. **Reference-based judge** (`utils.eval.run_global_judge`) — LLM-as-judge with the
   ground truth in the prompt, scoring generated *against* reference (dev-5 → full 72
   → cross-vendor).
3. **Stratified evaluation + judge robustness** (`utils.global_eval`) — results by
   shape type × modality, rubric-vs-judge correlation, cross-vendor Krippendorff-α.

The billed judge cells are guarded by **`RUN_JUDGE`** (default `False`). Flip it to
`True` in the bootstrap cell to actually call the API. The rubric and evaluation
cells are API-free and always run. `run_global_judge` is idempotent (skip-if-exists),
so re-running never double-charges.

In [8]:
from __future__ import annotations
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from utils import RESULTS_DIR, rubric, global_eval
from utils.eval import run_global_judge
from utils.llm import ask_text, ask_openai_text, OPENAI_JUDGE_MODEL_FINAL

# Judges
JUDGE_MODEL  = "claude-opus-4-8"        # primary judge (matches local track / NB05)
OPENAI_JUDGE = OPENAI_JUDGE_MODEL_FINAL  # cross-vendor judge (gpt-4o)
DEV5         = {"hr", "temp", "hum", "yr", "weekday"}  # one per shape type

# Safety switch: the judge cells below cost money. Set True to actually run them.
RUN_JUDGE = True

pd.set_option("display.width", 160)
print("rubric + eval ready | RUN_JUDGE =", RUN_JUDGE)

rubric + eval ready | RUN_JUDGE = True


## 1. Deterministic rubric (no LLM)

Scores every explanation against its ground truth and writes `results/global_rubric.csv`. Also flags any incomplete generations (missing `[EFFECT]`/`[IMPORTANCE]`/`[RECOMMENDATION]` sections).

In [9]:
rows = [rubric.score_result_file(p)
        for p in sorted((RESULTS_DIR / "global").glob("*.json"))]
rubric_df = pd.DataFrame(rows)
rubric_df.to_csv(RESULTS_DIR / "global_rubric.csv", index=False)

incomplete = rubric_df[rubric_df["incomplete"] != ""]
print(f"scored {len(rubric_df)} explanations | {len(incomplete)} incomplete")
if len(incomplete):
    display(incomplete[["form_pipeline", "xai_model", "feature", "incomplete"]])
rubric_df.head()

scored 72 explanations | 0 incomplete


,form_pipeline,xai_model,feature,form_type,incomplete,direction,rank,structure,total,parsed_rank,form
0,json,ebm,holiday,near-flat,,1.0,1.0,0.0,0.6667,9,near-flat
1,json,ebm,hr,categorical,,1.0,1.0,1.0,1.0000,1,categorical
2,json,ebm,hum,non-monotonic,,1.0,1.0,1.0,1.0000,5,non-monotonic
3,json,ebm,mnth,categorical,,1.0,1.0,1.0,1.0000,4,categorical
4,json,ebm,temp,non-monotonic,,1.0,1.0,1.0,1.0000,2,non-monotonic


### Rubric, stratified by shape type × modality (mean `total`)

In [10]:
global_eval.stratified_table(rubric_df, "total")

form_type,monotonic,non-monotonic,categorical,near-flat,overall
form_pipeline,,,,,
template,0.750,1.0,0.857,0.500,0.778
json,0.917,1.0,0.976,0.667,0.889
vision,1.000,1.0,1.000,0.633,0.898
tooluse,0.917,1.0,0.952,0.733,0.898


## 2. Reference-based judge — dev-5 (billed)

Runs the reference-based judge on the 5 dev features (`hr, temp, hum, yr, weekday`) across all modalities and models — the cheap validation set before the full run. Writes `results/global_judge/`.

In [11]:
if RUN_JUDGE:
    stems = [p.stem for p in (RESULTS_DIR / "global").glob("*.json")
             if p.stem.rsplit("_", 1)[-1] in DEV5]
    print(f"judging {len(stems)} dev-5 explanations with {JUDGE_MODEL} ...")
    dev_df = run_global_judge(ask_text, JUDGE_MODEL, only=stems)
    display(dev_df[["form_pipeline", "xai_model", "feature", "form_type",
                    "faithfulness", "clarity", "completeness"]])
else:
    print("skipped (RUN_JUDGE=False) — set RUN_JUDGE=True in the bootstrap cell")

judging 40 dev-5 explanations with claude-opus-4-8 ...


,form_pipeline,xai_model,feature,form_type,faithfulness,clarity,completeness
0,json,ebm,hr,categorical,5,4,5
1,json,ebm,hum,non-monotonic,5,4,5
2,json,ebm,temp,non-monotonic,5,4,5
3,json,ebm,weekday,near-flat,3,4,5
4,json,ebm,yr,monotonic,5,5,5
5,json,xgb,hr,categorical,5,3,5
6,json,xgb,hum,non-monotonic,5,4,5
7,json,xgb,temp,non-monotonic,3,3,5
8,json,xgb,weekday,categorical,4,4,5
9,json,xgb,yr,monotonic,5,4,5


## 2b. Reference-based judge — full run, all 72 (billed)

Judges every explanation. Idempotent: the dev-5 already scored above are skipped (no extra API calls).

In [12]:
if RUN_JUDGE:
    judge_df = run_global_judge(ask_text, JUDGE_MODEL)  # all 72
    print(f"judge records: {len(judge_df)}")
    display(judge_df[["form_pipeline", "xai_model", "feature", "form_type",
                      "faithfulness", "clarity", "completeness"]].head(12))
else:
    print("skipped (RUN_JUDGE=False)")

judge records: 72


,form_pipeline,xai_model,feature,form_type,faithfulness,clarity,completeness
0,json,ebm,holiday,near-flat,3,4,5
1,json,ebm,hr,categorical,5,4,5
2,json,ebm,hum,non-monotonic,5,4,5
3,json,ebm,mnth,categorical,5,4,5
4,json,ebm,temp,non-monotonic,5,4,5
5,json,ebm,weathersit,categorical,5,4,5
6,json,ebm,weekday,near-flat,3,4,5
7,json,ebm,windspeed,near-flat,4,5,5
8,json,ebm,yr,monotonic,5,5,5
9,json,xgb,holiday,near-flat,5,4,5


## 2c. Cross-vendor judge — OpenAI (billed)

A second vendor on the same explanations, for the Krippendorff-α robustness check. Needs `OPENAI_API_KEY` in `.env`. The adapter drops the Anthropic-only `cache_system` kwarg that `run_global_judge` passes.

In [13]:
def _ask_openai(prompt, *, system, model, max_tokens, cache_system=None, temperature=None):
    return ask_openai_text(prompt, system=system, model=model,
                           max_tokens=max_tokens, temperature=temperature)

if RUN_JUDGE:
    oai_df = run_global_judge(_ask_openai, OPENAI_JUDGE, out_subdir="global_judge_openai")
    print(f"OpenAI judge records: {len(oai_df)}")
else:
    print("skipped (RUN_JUDGE=False)")

OpenAI judge records: 72


## 3. Stratified evaluation + judge robustness

Merges rubric and judge scores, reports every table stratified by shape type × modality, the rubric-vs-judge correlation (validity of the LLM-free rubric), and cross-vendor Krippendorff-α (judge robustness). Runs on whatever is on disk — rubric-only until the judge cells above have been run.

In [14]:
rubric_df = global_eval.load_rubric()
judge_df  = global_eval.load_judge()      # None until the judge has run
merged    = global_eval.merge_scores(rubric_df, judge_df)

for name, tab in global_eval.report(merged).items():
    print(f"### {name}")
    display(tab)

if judge_df is not None:
    print("Rubric vs judge (faithfulness):",
          global_eval.rubric_judge_correlation(merged))
    vendors = sorted({p.name for p in RESULTS_DIR.glob("global_judge*") if p.is_dir()})
    if len(vendors) >= 2:
        for crit in global_eval.JUDGE_CRITERIA:
            print(f"cross-vendor alpha [{crit}]:",
                  round(global_eval.cross_vendor_alpha(vendors, crit), 3))
    else:
        print("(only one judge vendor — run cell 2c for cross-vendor alpha)")
else:
    print("(no judge output yet — run the cells above with RUN_JUDGE=True)")

### rubric_total


form_type,monotonic,non-monotonic,categorical,near-flat,overall
form_pipeline,,,,,
template,0.750,1.0,0.857,0.500,0.778
json,0.917,1.0,0.976,0.667,0.889
vision,1.000,1.0,1.000,0.633,0.898
tooluse,0.917,1.0,0.952,0.733,0.898


### judge_faithfulness


form_type,monotonic,non-monotonic,categorical,near-flat,overall
form_pipeline,,,,,
template,3.5,4.5,3.429,2.0,3.278
json,5.0,4.5,4.714,3.2,4.278
vision,5.0,5.0,4.571,3.4,4.389
tooluse,5.0,5.0,4.429,3.6,4.389


### judge_clarity


form_type,monotonic,non-monotonic,categorical,near-flat,overall
form_pipeline,,,,,
template,3.5,4.00,3.714,3.6,3.722
json,4.5,3.75,3.857,4.2,4.000
vision,3.0,3.25,3.714,3.6,3.500
tooluse,3.0,4.00,3.429,3.2,3.444


### judge_completeness


form_type,monotonic,non-monotonic,categorical,near-flat,overall
form_pipeline,,,,,
template,5.0,5.0,5.0,5.0,5.0
json,5.0,5.0,5.0,5.0,5.0
vision,5.0,5.0,5.0,5.0,5.0
tooluse,5.0,5.0,5.0,5.0,5.0


Rubric vs judge (faithfulness): {'n': 72, 'spearman_rho': 0.653, 'p_value': 0.0}
cross-vendor alpha [faithfulness]: 0.623
cross-vendor alpha [clarity]: 0.346
cross-vendor alpha [completeness]: -0.007
